# RDT-1B: Proprioceptive State Utilization Analysis

## Research Question
**Hypothesis**: Proprioceptive state information is NOT being fully utilized by the model.

## Experimental Design
1. **Pass normal state** → Measure gradient flow to state_adaptor
2. **Pass zeros (ablation)** → Measure gradient flow to state_adaptor  
3. **Compare**: If gradients barely change, state is underutilized

## What We'll Measure
- Gradient magnitude at state_adaptor (Linear layer)
- Gradient variance across batches
- Performance drop when state is zeroed
- Gradient sensitivity: ∂Loss/∂state_features

**Model**: RDT-1B (1.2B parameters)  
**State Encoder**: Single Linear layer (`state_adaptor`)  
**Key Metric**: If zero state → minimal gradient change = underutilization

---

## 1. Environment Setup

In [1]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab")
    !nvidia-smi
else:
    print("💻 Running locally")

🚀 Running on Google Colab
Sun Feb 15 03:47:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   31C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
# Install dependencies
!pip install -q torch torchvision transformers accelerate diffusers pillow numpy matplotlib seaborn scikit-learn

print("✅ Dependencies installed")

✅ Dependencies installed


### RDT-1B Specific Setup

**Official Repository**: `thu-ml/RoboticsDiffusionTransformer`

**Architecture**:
- Vision: SigLIP encoder (~400M params)
- Language: T5-XXL encoder (~11B params)
- State: `state_adaptor` Linear layer (input_dim = state_token_dim * 2 for state + mask)
- Action: Diffusion Transformer (1.2B params)

**Requirements**:
```bash
# Core dependencies (already installed above)
torch, transformers, diffusers

# Optional but recommended for speed
flash-attn  # Requires CUDA

# Model loading
pip install hydra-core omegaconf
```

**Model Files** (auto-downloaded on first run):
- T5-XXL encoder (~40GB)
- SigLIP vision encoder (~400MB)  
- RDT-1B checkpoint (~5GB)

**Note**: Due to model size (45GB+), this notebook assumes models are already cached or will download on demand via HuggingFace.

In [ ]:
# RDT-1B Environment Setup
# Clone official RDT repository for model definitions
import os

RDT_REPO_PATH = "RoboticsDiffusionTransformer"

if not os.path.exists(RDT_REPO_PATH):
    print("📥 Cloning RDT repository...")
    !git clone https://github.com/thu-ml/RoboticsDiffusionTransformer.git
    print("✅ RDT repository cloned")
else:
    print("✅ RDT repository already exists")

# Install RDT-specific dependencies
print("\n📦 Installing RDT dependencies...")
!pip install -q hydra-core omegaconf einops timm

# Optional: Install flash-attention for faster inference (requires CUDA)
try:
    import flash_attn
    print("✅ flash-attention already installed")
except ImportError:
    print("⚠️ flash-attention not installed (optional, speeds up inference)")
    print("   To install: pip install flash-attn --no-build-isolation")

print("\n✅ RDT environment ready")

In [8]:
!ls

sample_data


In [ ]:
# Clone repository if on Colab
if IN_COLAB:
    !git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
    %cd EmbodiedLLM
    !git checkout AnalyseMultipleHooks
    %cd MultipleHooksStudy
else:
    import os
    if not os.path.exists('hooks'):
        print("⚠️ Please run this notebook from MultipleHooksStudy directory")
    else:
        print("✅ Hook files found")

Cloning into 'EmbodiedLLM'...
remote: Enumerating objects: 528, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 528 (delta 43), reused 104 (delta 35), pack-reused 412 (from 2)
Receiving objects: 100% (528/528), 141.86 MiB | 17.87 MiB/s, done.
Resolving deltas: 100% (52/52), done.
[Errno 2] No such file or directory: 'EmbodiedLLM/MultipleHooksStudy'
/content


## 2. Import Hook Framework

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add hooks to path
sys.path.insert(0, str(Path.cwd() / 'hooks'))

from hooks.model_specific.rdt_hooks import RDTHooks

print("✅ Hook framework imported")

## 3. Load RDT-1B Model

**Official Loading Method** (Recommended):
```python
# Use RDTRunner from official repo
sys.path.insert(0, 'RoboticsDiffusionTransformer')
from rdt import RDTRunner

# Load with config
runner = RDTRunner.from_pretrained("rdt-1b")
model = runner.model
```

**Model Architecture**:
- `vision_encoder`: SigLIP (processes images)
- `language_encoder`: T5-XXL (processes text instructions)
- `state_adaptor`: Single Linear layer (state_token_dim * 2 → embedding_dim)
  - Input: Concatenated [robot_state, mask_indicator]
  - Output: State embedding for diffusion transformer
- `transformer`: Diffusion transformer for action prediction

**Note**: This notebook demonstrates hook attachment. For production use, follow [official RDT docs](https://github.com/thu-ml/RoboticsDiffusionTransformer).

In [ ]:
import os
import sys

# Clone the official RDT repository
repo_dir = os.path.expanduser("~/rdt_repo")
if not os.path.exists(repo_dir):
    print("Cloning RDT repository...")
    !git clone https://github.com/thu-ml/RoboticsDiffusionTransformer.git {repo_dir}
    print("✅ Repository cloned")
else:
    print(f"✅ Repository already exists at {repo_dir}")

# Add RDT repo to path
sys.path.insert(0, repo_dir)

# Install RDT requirements
print("\nInstalling RDT requirements...")
!pip install -q -r {repo_dir}/requirements.txt
!pip install -q flash-attn --no-build-isolation
print("✅ Requirements installed")

In [ ]:
import torch
from huggingface_hub import snapshot_download
from scripts.agilex_model import RoboticDiffusionTransformerModel
from omegaconf import OmegaConf

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download pretrained model checkpoint from HuggingFace
print("\nDownloading RDT-1B checkpoint from HuggingFace...")
checkpoint_dir = snapshot_download(
    repo_id="robotics-diffusion-transformer/rdt-1b",
    repo_type="model"
)
print(f"✅ Checkpoint downloaded to {checkpoint_dir}")

# Load config
config_path = os.path.join(repo_dir, "configs/base.yaml")
config = OmegaConf.load(config_path)

# Create RDT model instance using official code
print("\nCreating RDT model...")
model = RoboticDiffusionTransformerModel(
    config=config.model,
    pretrained_model_name_or_path=checkpoint_dir,
    pretrained_text_encoder_name_or_path="google/t5-v1_1-xxl",
    pretrained_vision_encoder_name_or_path="google/siglip-so400m-patch14-384",
)

print(f"✅ RDT-1B loaded successfully")
print(f"Model size: {sum(p.numel() for p in model.model.parameters()) / 1e9:.2f}B parameters")

## 4. Discover Model Structure

Verify that we correctly identify the `state_adaptor` (single Linear layer).

In [ ]:
# Initialize hook manager
hook_manager = RDTHooks(model)

# Discover structure
structure = hook_manager.discover_model_structure()

print("\n" + "="*60)
print("RDT-1B Model Structure Discovery")
print("="*60)
print(f"Model Name: {structure['model_name']}")
print(f"Has Proprio Encoder: {structure['has_proprio_encoder']}")
print(f"Proprio Encoder Type: {structure['proprio_encoder_type']}")
print(f"\nComponents Found:")
for component, attr in structure['components'].items():
    print(f"  - {component}: {attr}")

# Show state_adaptor details if found
if 'proprio_encoder_architecture' in structure:
    print(f"\nState Adaptor Architecture:")
    print(f"  Type: {structure['proprio_encoder_architecture']}")
    if 'proprio_input_dim' in structure:
        print(f"  Input dim: {structure['proprio_input_dim']}")
        print(f"  Output dim: {structure['proprio_output_dim']}")

print("="*60)

## 5. Attach Analysis Hooks

In [ ]:
# Attach ONLY gradient hooks (focus on state_adaptor)
print("Attaching gradient hooks...")

hook_manager.attach_gradient_hooks()
print("✓ Gradient flow hooks attached")
print("  → Monitoring state_adaptor specifically")

print("\n✅ Ready to analyze state utilization!")

## 6. Prepare Sample Data

RDT requires vision, language, AND proprioceptive state.

In [ ]:
from PIL import Image

# Sample image
sample_image = Image.new('RGB', (224, 224), color='green')

# Sample instruction
sample_instruction = "Grasp the green object and move it forward."

# Sample robot state (7-DOF: position + orientation)
# In real usage, this would come from robot sensors
sample_state = torch.randn(1, 7).to(device).half()  # Match model dtype

print("✅ Sample data prepared")
print(f"Image shape: {sample_image.size}")
print(f"Instruction: {sample_instruction}")
print(f"Robot state shape: {sample_state.shape}")
print(f"Robot state (sample): {sample_state[0, :3].cpu().numpy()}...")

## 7. Process Inputs

Convert raw data into model-ready format.

In [ ]:
# This is simplified - actual RDT input processing may differ
# Check the model's processor documentation for exact format

# For demonstration, we'll create mock inputs matching expected structure
# In practice, use the model's official processor

inputs = {
    'pixel_values': torch.randn(1, 3, 224, 224).to(device).half(),
    'input_ids': tokenizer(sample_instruction, return_tensors='pt')['input_ids'].to(device),
    'robot_state': sample_state  # Proprioceptive state
}

print("✅ Inputs prepared")
for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

## 8. Run Forward and Backward Pass

In [ ]:
# Set model to training mode
model.train()

# Forward pass
print("Running forward pass...")
try:
    outputs = model(**inputs)
    
    # RDT outputs action predictions (diffusion-based)
    # Extract appropriate output for loss computation
    if hasattr(outputs, 'action_pred'):
        action_pred = outputs.action_pred
    elif isinstance(outputs, dict) and 'action_pred' in outputs:
        action_pred = outputs['action_pred']
    else:
        # Fallback: use first tensor output
        action_pred = outputs[0] if isinstance(outputs, tuple) else outputs
    
    # Dummy loss for backward pass
    loss = action_pred.mean()
    
    print("Running backward pass...")
    loss.backward()
    
    print("✅ Forward and backward pass completed")
    print(f"Loss: {loss.item():.4f}")
    print(f"Action prediction shape: {action_pred.shape}")
    
except Exception as e:
    print(f"⚠️ Error during forward/backward pass: {e}")
    print("This may happen if model expects specific input format.")
    print("Check model documentation for exact input requirements.")

## 9. Baseline: Gradient Flow with Normal State

In [ ]:
# Capture baseline gradient flow
results_baseline = hook_manager.get_results()
gradient_baseline = results_baseline.get('gradient_flow', {})

print("\n" + "="*60)
print("BASELINE: Gradient Flow with Normal State")
print("="*60)

# Extract state_adaptor gradients
baseline_state_grad = None
if 'layer_profiles' in gradient_baseline:
    layer_profiles = gradient_baseline['layer_profiles']
    if 'state_adaptor' in layer_profiles:
        baseline_state_grad = layer_profiles['state_adaptor']
        print("\n🎯 State Adaptor Gradients (NORMAL STATE):")
        print(f"  Gradient norm: {baseline_state_grad.get('norm', 0):.6f}")
        print(f"  Gradient mean: {baseline_state_grad.get('mean_gradient', 0):.6f}")
        print(f"  Gradient variance: {baseline_state_grad.get('gradient_variance', 0):.6f}")

print("="*60)

## 10. Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Zero the OUTPUT of `state_adaptor`, not the input. 

This directly tests: "What if the state encoder contributed nothing?"

In [ ]:
# Create hook to zero out state_adaptor's output
ablation_handle = None

def zero_output_hook(module, input, output):
    """Replace state_adaptor output with zeros"""
    return torch.zeros_like(output)

# Find and hook the state_adaptor
for name, module in model.named_modules():
    if 'state_adaptor' in name or 'state_adapter' in name:
        ablation_handle = module.register_forward_hook(zero_output_hook)
        print(f"✓ Attached ablation hook to: {name}")
        break

print("Running ablation (state_adaptor OUTPUT = zeros)...")
model.train()
hook_manager.reset()
model.zero_grad()

# Forward + backward with same inputs, but state_adaptor outputs zeros
outputs_ablated = model(**inputs)
loss_ablated = outputs_ablated.mean()
loss_ablated.backward()

# Remove ablation hook
if ablation_handle:
    ablation_handle.remove()

print(f"✓ Ablation complete - state encoder contribution removed")

# Capture ablation gradients
results_ablated = hook_manager.get_results()
gradient_ablated = results_ablated.get('gradient_flow', {})

## 11. Analysis: Is State Being Utilized?

**Key Question**: Do gradients to state_adaptor change significantly when we zero the state?

In [ ]:
# Extract ablation gradients
ablated_state_grad = None
if 'layer_profiles' in gradient_ablated:
    layer_profiles = gradient_ablated['layer_profiles']
    if 'state_adaptor' in layer_profiles:
        ablated_state_grad = layer_profiles['state_adaptor']

# Compare baseline vs ablation
print("\n" + "="*60)
print("COMPARISON: Normal vs Ablated (Output = Zeros)")
print("="*60)

if baseline_state_grad and ablated_state_grad:
    baseline_norm = baseline_state_grad.get('norm', 0)
    ablated_norm = ablated_state_grad.get('norm', 0)
    
    grad_change = abs(baseline_norm - ablated_norm)
    grad_change_pct = (grad_change / baseline_norm * 100) if baseline_norm > 0 else 0
    
    print("\n🎯 State Adaptor Gradient Comparison:")
    print(f"  Normal (encoder active):     {baseline_norm:.6f}")
    print(f"  Ablated (output = zeros):    {ablated_norm:.6f}")
    print(f"  Absolute change:             {grad_change:.6f}")
    print(f"  Relative change:             {grad_change_pct:.2f}%")
    
    print("\n📊 VERDICT:")
    if grad_change_pct < 10:
        print("  ❌ UNDERUTILIZED: < 10% gradient change")
        print("     → Model barely uses state encoder output!")
        print("     → State encoder could be much simpler (or removed)")
    elif grad_change_pct < 30:
        print("  ⚠️  PARTIALLY UTILIZED: 10-30% gradient change")
        print("     → State has some influence but not critical")
    else:
        print("  ✅ WELL UTILIZED: > 30% gradient change")
        print("     → Model relies significantly on state encoder output")
    
    print("\n  Methodology: OUTPUT ABLATION")
    print("  We zeroed state_adaptor's output, not its input.")
    print("  This measures direct contribution to the model.")
    
    # Additional metrics
    baseline_mean = baseline_state_grad.get('mean_gradient', 0)
    ablated_mean = ablated_state_grad.get('mean_gradient', 0)
    mean_change_pct = abs(baseline_mean - ablated_mean) / abs(baseline_mean) * 100 if baseline_mean != 0 else 0
    
    print(f"\n  Gradient mean change: {mean_change_pct:.2f}%")
    
    baseline_var = baseline_state_grad.get('gradient_variance', 0)
    ablated_var = ablated_state_grad.get('gradient_variance', 0)
    var_change_pct = abs(baseline_var - ablated_var) / baseline_var * 100 if baseline_var > 0 else 0
    
    print(f"  Gradient variance change: {var_change_pct:.2f}%")

else:
    print("⚠️ Could not extract state_adaptor gradients")
    print("   Check that hook_manager correctly identifies state_adaptor")

print("="*60)

## 12. Visualization: Gradient Sensitivity

In [ ]:
# Visualize gradient comparison
if baseline_state_grad and ablated_state_grad:
    metrics = ['Gradient Norm', 'Mean Gradient', 'Gradient Variance']
    baseline_values = [
        baseline_state_grad.get('norm', 0),
        baseline_state_grad.get('mean_gradient', 0),
        baseline_state_grad.get('gradient_variance', 0)
    ]
    ablated_values = [
        ablated_state_grad.get('norm', 0),
        ablated_state_grad.get('mean_gradient', 0),
        ablated_state_grad.get('gradient_variance', 0)
    ]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, baseline_values, width, label='Normal State', color='#2ecc71')
    bars2 = ax.bar(x + width/2, ablated_values, width, label='Zeroed State', color='#e74c3c')
    
    ax.set_ylabel('Gradient Magnitude', fontsize=12)
    ax.set_title('State Adaptor: Gradient Sensitivity to Proprioceptive Input', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage change labels
    for i, (b, a) in enumerate(zip(baseline_values, ablated_values)):
        change_pct = abs(b - a) / b * 100 if b != 0 else 0
        ax.text(i, max(b, a), f'{change_pct:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Visualization: If bars are similar height → state underutilized")
else:
    print("⚠️ Cannot visualize - missing gradient data")

## 13. Export Results

In [ ]:
import json
from datetime import datetime

# Prepare focused export data
export_data = {
    'model_name': 'rdt-1b',
    'research_question': 'Is proprioceptive state being fully utilized?',
    'timestamp': datetime.now().isoformat(),
    'baseline': {
        'loss': loss.item() if 'loss' in locals() else None,
        'state_adaptor_gradients': baseline_state_grad if baseline_state_grad else {}
    },
    'ablation_zeroed_state': {
        'loss': loss_ablated.item() if 'loss_ablated' in locals() else None,
        'state_adaptor_gradients': ablated_state_grad if ablated_state_grad else {}
    },
    'analysis': {
        'gradient_norm_change_pct': grad_change_pct if 'grad_change_pct' in locals() else None,
        'verdict': 'UNDERUTILIZED' if 'grad_change_pct' in locals() and grad_change_pct < 10 else 
                   'PARTIAL' if 'grad_change_pct' in locals() and grad_change_pct < 30 else 
                   'WELL_UTILIZED' if 'grad_change_pct' in locals() else 'UNKNOWN'
    }
}

# Save to file
output_path = 'rdt_state_utilization_analysis.json'
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"✅ Results exported to {output_path}")

# Download if on Colab
if IN_COLAB:
    from google.colab import files
    files.download(output_path)
    print("📥 Results downloaded")

## 14. Summary

### Research Question
**Is proprioceptive state information being fully utilized in RDT-1B?**

### Methodology
1. ✅ Loaded RDT-1B model (1.2B params)
2. ✅ Identified state_adaptor (single Linear layer)
3. ✅ Attached gradient hooks to state_adaptor
4. ✅ Measured gradients with **normal state**
5. ✅ Measured gradients with **zeroed state** (ablation)
6. ✅ Compared gradient changes

### Key Metrics
- **Gradient norm change**: How much do gradients change when state is zeroed?
- **< 10% change** = State is underutilized (model doesn't rely on it)
- **> 30% change** = State is critical (model depends on it)

### Interpretation
If the hypothesis is TRUE (state underutilized):
- Model relies primarily on vision/language
- State encoder could be simplified or removed
- Opportunity for architectural improvement
- May explain why simple Linear layer is sufficient

If the hypothesis is FALSE (state well-utilized):
- Model genuinely uses proprioceptive information
- Current architecture is appropriate
- State information is integrated effectively

### Next Steps
1. Run on real robot demonstration data
2. Test across multiple tasks/domains
3. Compare different state dimensions (position vs orientation)
4. Measure utilization at different training checkpoints

---

**Repository**: [EmbodiedLLM/MultipleHooksStudy](https://github.com/PrithviTM-glitch/EmbodiedLLM)